## 4. 文档分块

整篇文档只生成一个向量会混合多个主题。标题感知分块先保存标题层级，再处理超长章节。

### 4.1. 模拟 `MarkdownHeaderTextSplitter`

`split_by_markdown_headers` 模拟 LangChain `MarkdownHeaderTextSplitter` 的核心行为：

- 识别 `#`、`##`、`###` 标题，并在标题处划分章节。
- 子章节继承父级标题路径；进入新的上级章节时，清除旧的下级标题。
- 每段同时返回正文 `text` 和标题路径 `metadata`。

这里使用普通字典展示中间结果；LangChain 返回 `Document`。本实现只处理三级规范 Markdown 标题。

In [1]:
import re
from pathlib import Path

HEADER_KEYS = {1: "title", 2: "section", 3: "subsection"}


def split_by_markdown_headers(text):
    sections = []
    metadata = {}
    lines = []
    in_fence = False

    def save_section():
        content = "\n".join(lines).strip()
        body_lines = lines[1:] if lines and re.match(r"^ {0,3}#{1,3}\s+", lines[0]) else lines
        has_body = any(line.strip() and line.strip() != "---" for line in body_lines)
        if content and has_body:
            sections.append({"text": content, "metadata": metadata.copy()})

    for line in text.splitlines():
        if re.match(r"^ {0,3}(```|~~~)", line):
            in_fence = not in_fence
            lines.append(line)
            continue

        match = None if in_fence else re.match(r"^ {0,3}(#{1,3})\s+(.+)$", line)
        if not match:
            lines.append(line)
            continue

        save_section()
        lines = [line]
        level = len(match.group(1))
        metadata[HEADER_KEYS[level]] = match.group(2).strip()
        for deeper_level in range(level + 1, 4):
            metadata.pop(HEADER_KEYS[deeper_level], None)

    save_section()
    return sections


### 4.2. 模拟 `RecursiveCharacterTextSplitter`

`split_long_text` 模拟 LangChain `RecursiveCharacterTextSplitter` 的核心行为：按分隔符优先级递归切分超长文本。

分隔符顺序是空行 → 换行 → 句号 → 分号 → 逗号 → 空格。先尝试保留较完整的自然段和句子；仍超过 800 字符时继续使用下一级分隔符，全部无效时按字符截断。

`split_markdown` 串联两步：先按标题分章，再限制每章长度；切出的子片段继承标题 `metadata`。本实现固定 `chunk_overlap=0`，未实现 LangChain 的重叠、长度函数等完整参数。

In [2]:
def split_long_text(text, max_chars=800, separators=("\n\n", "\n", "。", "；", "，", " ")):
    if len(text) <= max_chars:
        return [text]
    if not separators:
        return [text[index:index + max_chars] for index in range(0, len(text), max_chars)]

    separator, *remaining = separators
    if separator not in text:
        return split_long_text(text, max_chars, tuple(remaining))

    raw_parts = text.split(separator)
    parts = [
        part + separator if index < len(raw_parts) - 1 else part
        for index, part in enumerate(raw_parts)
    ]
    chunks = []
    current = ""
    for part in parts:
        candidate = current + part
        if len(candidate) <= max_chars:
            current = candidate
            continue
        if current:
            chunks.append(current)
        if len(part) <= max_chars:
            current = part
        else:
            chunks.extend(split_long_text(part, max_chars, tuple(remaining)))
            current = ""
    if current:
        chunks.append(current)
    return [chunk.strip() for chunk in chunks if chunk.strip()]


def split_markdown(text, max_chars=800):
    chunks = []
    for section in split_by_markdown_headers(text):
        for part in split_long_text(section["text"], max_chars):
            chunks.append({"text": part, "metadata": section["metadata"].copy()})
    return chunks


### 4.3. 检查分块结果

In [3]:
path = Path("../data/产品/瑜伽裤-YG301/质检报告.md")
text = path.read_text(encoding="utf-8")
header_sections = split_by_markdown_headers(text)
chunks = split_markdown(text)

print(f"原文字符数：{len(text)}")
print(f"标题章节数：{len(header_sections)}")
print(f"最终片段数：{len(chunks)}")
for index, chunk in enumerate(chunks, start=1):
    path_text = " > ".join(chunk["metadata"].values()) or "无标题"
    print(f"\n--- 片段 {index}（{len(chunk['text'])} 字符）---")
    print(f"标题路径：{path_text}")
    print(chunk["text"][:260])


原文字符数：2928
标题章节数：9
最终片段数：12

--- 片段 1（187 字符）---
标题路径：SGS 质检报告 - 瑜伽裤 SKU-YG301
# SGS 质检报告 - 瑜伽裤 SKU-YG301

> 报告编号: SGS-RPT-YG301-2026
> 商品代号: SKU-YG301
> 检测机构: SGS CSTC Technical Co., Ltd. (Shenzhen Branch)
> 报告日期: 2026-06-20
> 测试样品: M码, 3件 (A/B/C)
> 状态: Active

---

--- 片段 2（10 字符）---
标题路径：SGS 质检报告 - 瑜伽裤 SKU-YG301 > 1. 测试摘要
## 1. 测试摘要

--- 片段 3（753 字符）---
标题路径：SGS 质检报告 - 瑜伽裤 SKU-YG301 > 1. 测试摘要
| 序号 | 测试项目 | 测试方法 | 判定标准 | 结果 | 判定 |
| :---: | :--- | :--- | :--- | :---: | :---: |
| 1 | 防透光 | SGS Squat-Proof | 透光率≤5% | ≤2% (5级) | PASS |
| 2 | 抗起球 | ASTM D3512 | ≥3.5级 | 4.5级 | PASS |
| 3 | 弹性恢复 | 500次循环拉伸 | 恢复率≥90% | ≥95% | PASS |
| 4 | 耐光色牢度 | AATCC 16 

--- 片段 4（50 字符）---
标题路径：SGS 质检报告 - 瑜伽裤 SKU-YG301 > 1. 测试摘要
| 14 | UPF防紫外线 | AATCC 183 | UPF≥40 | 50+ | PASS |

--- 片段 5（30 字符）---
标题路径：SGS 质检报告 - 瑜伽裤 SKU-YG301 > 1. 测试摘要
综合判定: **PASS** — 14项全部通过。

---

--- 片段 6（331 字符）---
标题路径：SGS 质检报告 - 瑜伽裤 SKU-YG301 > 2. 详细测试结果 > 2.1 防透光 (SGS Squat-Proof)
### 2.1 防透光 (SGS Squat-Proof)

| 样品 | 状态 | 透光率 

该实现识别规范的 Markdown 标题，限长时可能拆开表格或列表。`max_chars` 应根据文档结构和检索评测调整。